# Cone-linearized annulus-disc analysis
**The experiment.** Every natural-image annulus patch is shown three ways, interleaved:

| `stimulusTag` | what it is |
|---|---|
| `image` | the image patch itself |
| `intensity` | uniform annulus at the linear-equivalent intensity (patch averaged over the surround RF) |
| `linConeIntensity` / `lin cone intensity` | uniform annulus at the **cone-linearized** equivalent intensity — averaged after a Weber cone nonlinearity `I / (I + WeberConstant)` |

Comparing image against each annulus asks how much of the cell's preference for the real image
survives when the averaging happens in cone-response space instead of intensity space. The
measure is the nonlinearity index per patch, exactly as `computeNLI` defines it:

$$\mathrm{NLI} = \frac{\mathrm{image} - \mathrm{disc}}{|\mathrm{image}| + |\mathrm{disc}|}$$

set to zero when neither response clears a recording-mode threshold.


In [1]:
import contextlib
import io
import sys
import time

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        f'This notebook requires the retinanalysis Python 3.11 kernel; '
        f'got Python {sys.version.split()[0]} at {sys.executable}')
import_started = time.perf_counter()

import retinanalysis as ra
import numpy as np
import pandas as pd

from retinanalysis.SCutils import explore as sc
from retinanalysis.SCutils.protocols import linear_equivalent_disc as led

print(f'Python {sys.version.split()[0]} | {sys.executable}')
print(f'Imports ready in {time.perf_counter() - import_started:.2f} s')

Python 3.11.13 | /Users/chrischen/opt/anaconda3/envs/retinanalysis/bin/python
Imports ready in 0.69 s


## 1. Find annulus-disc experiment dates and cells

The fast table shows every available `LinearEquivalentAnnulus` cell × resolved
`onlineAnalysis` × numeric FilterWheel combination, together with the short cell type and
protocol name. Its values can be copied directly into Section 2. Detailed raw response
loading is deferred until one condition is selected.


In [ ]:
ANNULUS_PROTOCOLS = ('LinearEquivalentAnnulus',)
PROTOCOL_LABEL = 'cone-linearized annulus disc'

protocol_blocks = led.find_blocks(protocols=ANNULUS_PROTOCOLS, show=False)
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    protocol_blocks = led.check_series_resistance(
        protocol_blocks, show=False, sample_series_resistance=True)
protocol_cells = (protocol_blocks[[
    'exp_name', 'cell_label', 'cell_type_short', 'onlineAnalysis',
    'filter_wheel_ndf', 'protocol']]
    .drop_duplicates()
    .sort_values(['exp_name', 'cell_label', 'onlineAnalysis',
                  'filter_wheel_ndf', 'protocol'])
    .reset_index(drop=True))
protocol_cells.insert(
    0, 'date_index', pd.factorize(protocol_cells['exp_name'], sort=False)[0] + 1)
print(f'{PROTOCOL_LABEL}: {len(protocol_cells)} available conditions across '
      f'{protocol_cells.exp_name.nunique()} experiments')
sc.scroll_table(protocol_cells, height=420, num_cols=('filter_wheel_ndf',))


## 2. Analyze one cell condition

This section is standalone after the import cell: enter a `date_index` from Section 1 (or a
previously known index), cell label, resolved `onlineAnalysis`, and numeric FilterWheel value.
It performs its own date lookup and detailed block loading, so Section 1 does not need to be run
first. The first output lists the selected cell label/type, matching block IDs, annulus inner
and outer diameters, surround RF sigma,
stimulus timing, and one row per `imageName`, including its epoch count and
`meanIntensity = maxIntensity × backgroundIntensity`.

Responses use the exact onset window from `preTime` through `preTime + stimTime`: spike count
for extracellular recordings and baseline-subtracted signed area (pA·s) for whole-cell
recordings. Each scatter point is one `(imageName, patchIndex)` pair; x and y error bars are
SEM across repeat epochs. The per-image and pooled figures show image vs standard annulus in grey
and image vs cone-linearized annulus in red, with a unity line. Patch indices that restart in a new
image remain separate. The final figure uses empirical NLI CDFs and a second panel with
the mean ± SEM across every image-specific patch for this cell. After Section 2a has saved this
exact block set once, rerunning after a kernel restart reloads those patch results instead of
re-reading traces and detecting spikes. Pass `reuse_saved=False` to `analyze_condition` when a
deliberate raw-data recomputation is needed. The optional sanity-check call shows a few
response-selected `(imageName, patchIndex)` examples: trial-normalized PSTHs for
extracellular recordings, or baseline-subtracted mean traces for `exc`/`inh`.

In [ ]:
# Standalone after the imports; Section 1 is optional.
ANNULUS_PROTOCOLS = ('LinearEquivalentAnnulus',)
PROTOCOL_LABEL = 'cone-linearized annulus disc'
DATE_INDEX = 39
CELL_LABEL = 'Cell7'
ONLINE_ANALYSIS = 'extracellular'  # 'extracellular', 'exc', or 'inh'
FILTER_WHEEL_VALUE = 1.0          # authoritative numeric FilterWheel value

analysis_dates = led.find_protocol_cells(ANNULUS_PROTOCOLS, show=False)
analysis_dates.insert(
    0, 'date_index', pd.factorize(analysis_dates['exp_name'], sort=False)[0] + 1)
date_rows = analysis_dates.loc[analysis_dates.date_index.eq(DATE_INDEX)]
if date_rows.empty:
    raise ValueError(f'date_index {DATE_INDEX} is not available for {PROTOCOL_LABEL}')
EXP_NAME = date_rows.exp_name.iloc[0]

df_blocks = led.find_blocks(
    exp_names=[EXP_NAME], protocols=ANNULUS_PROTOCOLS, show=False)
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    df_blocks = led.check_series_resistance(
        df_blocks, show=False, sample_series_resistance=True)
selected_blocks = df_blocks.copy()
if selected_blocks.empty:
    raise ValueError(f'{EXP_NAME!r} has no cone-linearized annulus-disc blocks')

condition_blocks = led.select_condition_blocks(
    selected_blocks, CELL_LABEL, ONLINE_ANALYSIS, FILTER_WHEEL_VALUE)
key_parameters = (condition_blocks[[
    'annulusInnerDiameter', 'annulusOuterDiameter', 'rfSigmaSurround',
    'preTime', 'stimTime', 'tailTime']]
    .drop_duplicates().reset_index(drop=True))
print('Key protocol parameters:')
print(key_parameters.to_string(index=False))
condition = led.analyze_condition(condition_blocks)
image_fig, pooled_fig, nli_fig = led.plot_condition(condition)
# sample_response_fig = led.plot_condition_sample_psths(
#     condition_blocks, condition, n_pairs=3)


### 2a. Save this cell for population analysis

Save one compressed HDF5 record for the selected cell condition, similar to a MATLAB struct
containing arrays. Cell metadata is stored once, and the image summary and patch responses are
kept as typed arrays instead of tens of thousands of CSV rows. `condition_population_table()`
still expands the selected condition for display when needed. Each distinct date/cell/mode/FW
condition gets its own file. Before saving, the code checks all routed and legacy outputs for
the same date/cell/mode/site/FW condition, prints an alert, replaces the canonical record, and
removes duplicate copies. The file also becomes the fast restart source for Section 2 when the selected
block IDs match. Later,
`led.load_condition_outputs()` combines every saved cell into one population table.

In [ ]:
condition_output_path = led.save_condition_output(condition, remove_duplicates=True)
population_ready = led.condition_population_table(condition)
sc.scroll_table(
    population_ready, height=360,
    num_cols=('filter_wheel_ndf', 'patchIndex', 'maxIntensity',
              'meanIntensity', 'image_response', 'disc_response',
              'cone_disc_response', 'nli_image_vs_disc',
              'nli_image_vs_cone_disc'));

### 2b. Check saved cells

Read only the small metadata attributes from saved annulus-disc conditions—without loading the
patch arrays—and list date, cell label/type, resolved `onlineAnalysis`, and FilterWheel value.

In [ ]:
saved_cells = led.load_condition_index(protocol=ANNULUS_PROTOCOLS)
print(f'{len(saved_cells)} saved cell condition(s)')
sc.scroll_table(saved_cells, height=300, num_cols=('filter_wheel_ndf',));


## 3. Population by image and cell type

Load saved HDF5 conditions for `LinearEquivalentAnnulus`. Each row is one cell ×
FilterWheel × `imageName`: `meanIntensity` is x, and the two y values are mean patch NLI for
the ordinary and cone-linearized annulus comparisons. Then group rows into `[500, 1500)`,
`[1500, 3500)`, `[3500, 6000)`, and `[6000, 20000]` within each cell type and plot population
mean NLI ± SEM. No averaging occurs across cells before this grouping.


In [ ]:
image_nli = led.load_condition_image_nli_summary(
    protocol=ANNULUS_PROTOCOLS)
print(f'{len(image_nli)} cell/image rows from {image_nli.cell_id.nunique()} cells')
image_nli_view = image_nli[[
    'exp_name', 'cell_label', 'cell_type', 'onlineAnalysis', 'protocol',
    'filter_wheel_ndf', 'imageName', 'meanIntensity', 'n_patches',
    'mean_nli_disc', 'mean_nli_cone_disc']]
sc.scroll_table(
    image_nli_view, height=360,
    num_cols=('filter_wheel_ndf', 'meanIntensity', 'n_patches',
              'mean_nli_disc', 'mean_nli_cone_disc'))
light_level_nli = led.summarize_image_nli_light_levels(image_nli)
print(f'{light_level_nli.n_cell_images.sum()} of {len(image_nli)} rows are within 500–20000')
sc.scroll_table(
    light_level_nli, height=300,
    num_cols=('light_min', 'light_max', 'meanIntensity', 'n_cells',
              'n_cell_images', 'mean_nli_disc', 'sem_nli_disc',
              'mean_nli_cone_disc', 'sem_nli_cone_disc'))
led.plot_image_nli_by_cell_type(
    image_nli, title_prefix='Cone-linearized annulus disc');


### 3a. Pooled patch NLI distributions

Pool every patch NLI from saved annulus-disc HDF5 conditions, without averaging by image or cell.
The first figure compares ordinary and cone-linearized annuli across all saved patches. The second
repeats the normalized 50-bin density and empirical CDF separately for each cell type.


In [ ]:
patch_nli = led.load_condition_patch_nli(
    protocol=ANNULUS_PROTOCOLS)
finite_patch_nli = np.isfinite(
    patch_nli[['nli_disc', 'nli_cone_disc']]).sum()
print(f"{finite_patch_nli['nli_disc']} standard and "
      f"{finite_patch_nli['nli_cone_disc']} cone-lin patch NLIs from "
      f'{len(patch_nli)} saved rows and {patch_nli.cell_id.nunique()} cells')
led.plot_pooled_patch_nli_distributions(
    patch_nli, bins=50, title_prefix='Cone-linearized annulus disc');
led.plot_patch_nli_distributions_by_cell_type(
    patch_nli, bins=50, title_prefix='Cone-linearized annulus disc');


### 3b. Cell-level averages by light level and paired high-light comparison

For each cell, pool all patches across all `imageName`s within `[500, 1500)` (~1k) or
`[6000, 20000]` (~10k), then calculate one ordinary-annulus and one cone-linearized mean NLI.
Faint points are those cell means; large points and error bars are population mean ± SEM
across cells, so a cell with more patches does not get extra population weight. Finally, for
each cell type, the paired plot pools every patch at 7000 R* and above into one ordinary-annulus
mean and one cone-linearized-annulus mean per cell, then joins those two values with a line.


In [ ]:
cell_light_nli = led.summarize_cell_patch_nli_light_levels(patch_nli)
sc.scroll_table(
    cell_light_nli, height=320,
    num_cols=('light_min', 'light_max', 'meanIntensity', 'n_images',
              'n_patches', 'mean_nli_disc', 'mean_nli_cone_disc'))
led.plot_cell_patch_nli_by_light(
    cell_light_nli, title_prefix='Cone-linearized annulus disc');
high_light_cell_nli = led.summarize_cell_patch_nli_above(
    patch_nli, min_intensity=7000)
sc.scroll_table(
    high_light_cell_nli, height=300,
    num_cols=('min_intensity', 'meanIntensity', 'n_images', 'n_patches',
              'mean_nli_disc', 'mean_nli_cone_disc'))
led.plot_cell_patch_nli_paired_above(
    high_light_cell_nli, min_intensity=7000,
    title_prefix='Cone-linearized annulus disc');


## 4. Inspect selected date metadata (optional)

This optional section uses the date already loaded in Section 2 and groups its recordings by
cell, resolved recording mode, annulus-disc site, and light setting. It is not required for the
single-cell or population analyses above.

In [ ]:
groups = led.group_blocks(selected_blocks)


### 4a. Preview recorded stimuli

This optional visualization uses the date already loaded in Section 2. The dropdown contains
every recorded `imageName` for that date; choosing one redraws several recorded patches as a
tilted flash sequence. Each triplet proceeds from natural-image annulus to linear-equivalent
annulus to cone-linearized annulus,
and the diagonal time arrow shows the repeated presentation order. Spatial x/y arrows are shown
on the first stimulus plane and follow its tilted edges. As in the original protocol, the full
canvas and masked regions remain at the recorded `backgroundIntensity`; only the annulus changes
between the natural-image patch, linear-equivalent intensity, and cone-linearized intensity. The
center spot is overlaid on every flash at `backgroundIntensity × (1 + centerSpotContrast)`. All
stimuli use a common grey scale. The image is loaded from the van Hateren resources in the turner
package the same way `NaturalImageFlashProtocol.m` does — big-endian `.iml`, rescaled so the
brightest pixel is 1, at 6.6 µm per image pixel.

A useful check that the geometry is right: the mean intensity inside the aperture should land on
the recorded `equivalentIntensity`, since that is what the equivalent disc is defined to be.

In [ ]:
image_example = led.stimulus_example_widget(selected_blocks, sequence_length=4)
image_example